# InSight: Google Colab Runner

Notebook ini menjalankan backend **FastAPI + Docling + Qdrant Local + SQLite** bersama model **Unsloth / Hugging Face** dengan pilihan presisi (`4bit`, `8bit`, atau `full`) sesuai paket GPU Colab Anda.

> **Catatan Privasi:** Runtime Colab berjalan di server Google (bukan lokal perangkat), tetapi dokumen Anda diproses langsung di runtime sesi Anda sendiri tanpa dikirim ke API LLM pihak ketiga dan akan hilang saat runtime dimatikan (kecuali Anda mengaktifkan penyimpanan ke Google Drive).

In [ ]:
# 1. Clone repository & install dependensi
![ -d InSight ] || git clone https://github.com/Cahyo-Wisanggeni/InSight.git
%cd InSight
!pip install -q -r requirements.txt
!pip install -q -r requirements-colab.txt

In [ ]:
# 2. Konfigurasi Runtime (sesuaikan dengan GPU Colab Anda)
import os

os.environ["INSIGHT_HOST"] = "0.0.0.0"
os.environ["INSIGHT_PORT"] = "8000"

# Pilih backend: "unsloth", "hf", atau "retrieval_only"
os.environ["INSIGHT_LLM_BACKEND"] = "unsloth"
os.environ["INSIGHT_LLM_MODEL"] = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

# Pilih presisi: "4bit" (T4 15GB), "8bit" (L4 24GB), atau "full" (A100 40GB+)
os.environ["INSIGHT_PRECISION"] = "4bit"

# Aktifkan reranker ringan (~22M-278M params) untuk Step 1 penyaringan konteks
os.environ["INSIGHT_USE_RERANKER"] = "true"

# Kata sandi sesi untuk melindungi URL tunnel publik (sangat disarankan diisi)
os.environ["INSIGHT_PASSWORD"] = "ganti-password-anda"

# Opsional: arahkan INSIGHT_DATA_DIR ke Google Drive agar Qdrant & SQLite tidak hilang
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["INSIGHT_DATA_DIR"] = "/content/drive/MyDrive/InSightData"

In [ ]:
# 3. Jalankan server FastAPI di background & buka tunnel Cloudflare
import subprocess, time, re
from pathlib import Path

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

server_log = open("/tmp/insight_server.log", "w")
server = subprocess.Popen(["python", "run.py"], stdout=server_log, stderr=subprocess.STDOUT)
time.sleep(3)

tunnel_log_path = Path("/tmp/cloudflared.log")
tunnel_log = open(tunnel_log_path, "w")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

url = None
for _ in range(30):
    text = tunnel_log_path.read_text(errors="ignore") if tunnel_log_path.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if match:
        url = match.group(0)
        break
    time.sleep(1)

if url:
    print("Buka InSight UI di:", url)
else:
    print("URL tunnel belum muncul. Cek /tmp/cloudflared.log dan /tmp/insight_server.log")